# 04 — SafeDrug Model

SafeDrug — drug recommendation model that explicitly minimizes DDI rate.
Unlike GAMENet/MoleRec which only maximize recommendation accuracy,
SafeDrug adds a DDI constraint to the loss function.

Key inputs: `ddi_adj` (DDI adjacency matrix) and `ddi_mask_H` — both
produced automatically by the drug recommendation task function.

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import drug_recommendation_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import SafeDrug, GAMENet
from pyhealth.trainer import Trainer
from pyhealth.metrics import ddi_rate_score
import pandas as pd

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(drug_recommendation_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
# SafeDrug — DDI-constrained recommendation
safedrug = SafeDrug(
    dataset=task_dataset,
    feature_keys=['conditions', 'drugs'],
    label_key='drugs',
    mode='multilabel',
    ddi_adj=task_dataset.ddi_adj,
    ddi_mask_H=task_dataset.ddi_mask_H,
)
sd_trainer = Trainer(model=safedrug, metrics=['jaccard', 'prauc', 'f1'])
sd_trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
                 epochs=50, monitor='jaccard')
sd_result = sd_trainer.evaluate(test_loader)
sd_ddi = ddi_rate_score(sd_result['y_prob'], task_dataset.ddi_adj)
print(f'SafeDrug — jaccard={sd_result["jaccard"]:.4f}, DDI rate={sd_ddi:.4f}')